# Preparación y comparación de modelos de regresión

## Ficha de formulación del problema

| Elemento | Definición |
|---|---|
| Problema de negocio | Estimar el gasto de compra elegible de clientes existentes durante los seis meses siguientes, para priorizar acciones de retención y planificación comercial. |
| Unidad de análisis | Un cliente identificado por `customer_id` y una ventana de observación; una fila por cliente en cada corte. |
| Predicción | Al cierre del 30 de junio de 2010 o del 30 de junio de 2011, predecir el importe agregado del periodo futuro correspondiente. |
| Variable objetivo (`y`) | Suma de `quantity × price` en el periodo futuro para líneas de compra con cantidad y precio positivos, excluyendo cancelaciones (`C`) y ajustes (`A`). Se asigna cero a clientes de la cohorte sin compras elegibles posteriores. |
| Variables predictoras (`X`) | Gasto, pedidos, unidades, productos distintos, días activos, recencia, gasto promedio por pedido, país más frecuente y conteos históricos de devoluciones, cancelaciones y ajustes. |
| Identificador | `customer_id` define la entidad, pero nunca entra como predictor. |
| Población | Clientes con ID y al menos una compra elegible en la ventana histórica. No se imputan identidades faltantes. |

El gasto calculado es un **proxy transaccional**, no una medida contable de ventas netas: no incluye costos, impuestos ni conciliación contable. Los importes de devoluciones, cancelaciones y ajustes no se incorporan a `y`; los conteos de esos eventos históricos sí se conservan como predictores.

## Diseño temporal

Se usan ventanas de observación de 181 días y horizontes objetivo comparables de 162 días:

| Conjunto | Observación (solo `X`) | Objetivo (solo `y`) |
|---|---|---|
| Entrenamiento | 2010-01-01 a 2010-06-30 | 2010-07-01 a 2010-12-09 |
| Prueba temporal | 2011-01-01 a 2011-06-30 | 2011-07-01 a 2011-12-09 |

El corte impide que una transacción futura al momento de predicción contribuya a los predictores. La prueba representa clientes existentes en un periodo posterior; clientes pueden aparecer en ambos conjuntos en años distintos, pero no se mezclan transacciones futuras en sus características. Al elegir un ganador con esta prueba, sus métricas son comparativas y no constituyen una evaluación final completamente independiente.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

relative_data_path = Path("03_ETL_DATA_QUALITY") / "online_retail_II_procesado.csv.gz"
data_candidates = [
    Path.cwd() / relative_data_path,
    Path.cwd().parent / relative_data_path,
]
DATA_FILE = next((path.resolve() for path in data_candidates if path.exists()), None)
if DATA_FILE is None:
    raise FileNotFoundError(
        "No se encontró 03_ETL_DATA_QUALITY/online_retail_II_procesado.csv.gz. "
        "Ejecuta el notebook desde la carpeta del proyecto o su subcarpeta 05_MACHINE_LEARNING."
    )

print(f"Archivo de entrada: {DATA_FILE}")
print(f"Directorio del proyecto: {DATA_FILE.parent.parent}")


Archivo de entrada: C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA\03_ETL_DATA_QUALITY\online_retail_II_procesado.csv.gz
Directorio del proyecto: C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA


In [2]:
required_columns = [
    "invoice", "stock_code", "quantity", "invoice_date", "price", "customer_id",
    "country", "source_sheet", "line_amount", "is_cancellation", "is_adjustment",
]
data = pd.read_csv(
    DATA_FILE,
    compression="gzip",
    usecols=required_columns + ["is_exact_duplicate"],
    parse_dates=["invoice_date"],
    dtype={
        "invoice": "string",
        "stock_code": "string",
        "country": "string",
        "source_sheet": "string",
        "is_cancellation": "boolean",
        "is_adjustment": "boolean",
        "is_exact_duplicate": "boolean",
    },
)
data["customer_id"] = pd.to_numeric(data["customer_id"], errors="coerce").astype("Int64")

assert not data.empty, "El archivo procesado no contiene registros."
assert data["invoice_date"].notna().all(), "Hay fechas no válidas en el archivo procesado."
print(f"Dimensiones cargadas: {data.shape[0]:,} filas × {data.shape[1]} columnas")
print(f"Rango de fechas: {data['invoice_date'].min()} — {data['invoice_date'].max()}")
display(data.head())


Dimensiones cargadas: 1,067,371 filas × 12 columnas
Rango de fechas: 2009-12-01 07:45:00 — 2011-12-09 12:50:00


,invoice,stock_code,quantity,invoice_date,price,customer_id,country,source_sheet,line_amount,is_cancellation,is_adjustment,is_exact_duplicate
0,489434,85048,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010,83.40,False,False,False
1,489434,79323P,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,81.00,False,False,False
2,489434,79323W,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,81.00,False,False,False
3,489434,22041,48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010,100.80,False,False,False
4,489434,21232,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010,30.00,False,False,False


## Reglas de selección y tratamiento

- Los faltantes de `customer_id` no se imputan: sin identidad no es posible construir una historia individual. Se cuantifican antes de excluirlos de las cohortes.
- Se retiene una sola copia de las filas que el ETL marcó como duplicadas exactamente **dentro de su hoja de origen**. Esto evita que diferencias de formato ya normalizadas por el ETL se confundan con duplicados originales; se reportan las filas excedentes y no se modifica el archivo ETL.
- Para gasto histórico y objetivo se consideran únicamente líneas con `quantity > 0`, `price > 0` e importe positivo que no sean cancelaciones ni ajustes. Devoluciones, cancelaciones, ajustes y compras no elegibles no se borran del archivo: los conteos históricos de eventos se convierten en características cuando tienen cliente identificado.
- Los clientes del objetivo se limitan a quienes tienen al menos una compra elegible en observación. Quienes no compren en el horizonte futuro reciben `y = 0`; así la predicción corresponde a gasto incondicional de esa cohorte, incluyendo inactividad futura.
- No se recortan valores atípicos: la transformación `log1p` del objetivo reduce la asimetría durante el ajuste, y las métricas se calculan en la escala original del importe.


In [3]:
original_rows = len(data)
missing_customer_rows = int(data["customer_id"].isna().sum())
duplicate_columns = [
    "source_sheet", "invoice", "stock_code", "quantity", "invoice_date",
    "price", "customer_id", "country",
]
duplicate_flag = data["is_exact_duplicate"].fillna(False).astype(bool)
nonduplicate_rows = data.loc[~duplicate_flag]
flagged_rows = data.loc[duplicate_flag].drop_duplicates(
    subset=duplicate_columns,
    keep="first",
)
model_data = pd.concat([nonduplicate_rows, flagged_rows]).sort_index().copy()
duplicate_excess = original_rows - len(model_data)
if duplicate_excess > int(duplicate_flag.sum()):
    raise AssertionError("El tratamiento excede las filas que el ETL marcó como duplicadas.")

def eligible_purchase_mask(frame):
    return (
        frame["quantity"].gt(0)
        & frame["price"].gt(0)
        & frame["line_amount"].gt(0)
        & ~frame["is_cancellation"].fillna(False).astype(bool)
        & ~frame["is_adjustment"].fillna(False).astype(bool)
    )

print(f"Filas originales leídas: {original_rows:,}")
print(f"Filas excedentes por duplicado exacto retenidas fuera del modelado: {duplicate_excess:,}")
print(f"Filas después de deduplicar para este análisis: {len(model_data):,}")
print(f"Filas sin customer_id (no identificables por cliente): {missing_customer_rows:,}")


Filas originales leídas: 1,067,371
Filas excedentes por duplicado exacto retenidas fuera del modelado: 12,133
Filas después de deduplicar para este análisis: 1,055,238
Filas sin customer_id (no identificables por cliente): 243,007


In [4]:
NUMERIC_FEATURES = [
    "historical_spend",
    "order_count",
    "units_purchased",
    "distinct_products",
    "active_days",
    "days_since_last_purchase",
    "mean_order_spend",
    "historical_return_lines",
    "historical_cancellation_lines",
    "historical_adjustment_lines",
]
CATEGORICAL_FEATURES = ["country"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

def make_customer_cohort(observation_start, observation_end, target_start, target_end):
    observation_start = pd.Timestamp(observation_start)
    observation_end = pd.Timestamp(observation_end)
    target_start = pd.Timestamp(target_start)
    target_end = pd.Timestamp(target_end)

    customer_rows = model_data.loc[model_data["customer_id"].notna()].copy()
    observed = customer_rows.loc[
        customer_rows["invoice_date"].ge(observation_start)
        & customer_rows["invoice_date"].lt(observation_end)
    ].copy()
    historical_purchases = observed.loc[eligible_purchase_mask(observed)].copy()

    if historical_purchases.empty:
        raise ValueError(f"No hay compras históricas elegibles desde {observation_start.date()}.")

    historical_purchases["purchase_day"] = historical_purchases["invoice_date"].dt.normalize()
    grouped = historical_purchases.groupby("customer_id", sort=False)
    features = grouped.agg(
        historical_spend=("line_amount", "sum"),
        order_count=("invoice", "nunique"),
        units_purchased=("quantity", "sum"),
        distinct_products=("stock_code", "nunique"),
        active_days=("purchase_day", "nunique"),
        last_purchase=("invoice_date", "max"),
    )
    features["days_since_last_purchase"] = (
        observation_end.normalize() - features.pop("last_purchase").dt.normalize()
    ).dt.days

    invoice_spend = (
        historical_purchases.groupby(["customer_id", "invoice"], sort=False)["line_amount"]
        .sum()
        .groupby(level="customer_id")
        .mean()
        .rename("mean_order_spend")
    )
    country_mode = grouped["country"].agg(
        lambda values: values.dropna().mode().iloc[0] if values.notna().any() else np.nan
    ).rename("country")
    features = features.join(invoice_spend).join(country_mode)

    observed_events = observed.assign(
        historical_return_lines=observed["quantity"].lt(0).astype("int8"),
        historical_cancellation_lines=observed["is_cancellation"].fillna(False).astype("int8"),
        historical_adjustment_lines=observed["is_adjustment"].fillna(False).astype("int8"),
    )
    event_features = observed_events.groupby("customer_id", sort=False)[
        [
            "historical_return_lines",
            "historical_cancellation_lines",
            "historical_adjustment_lines",
        ]
    ].sum()
    features = features.join(event_features, how="left")
    count_columns = [
        "historical_return_lines",
        "historical_cancellation_lines",
        "historical_adjustment_lines",
    ]
    features[count_columns] = features[count_columns].fillna(0)

    future = customer_rows.loc[
        customer_rows["invoice_date"].ge(target_start)
        & customer_rows["invoice_date"].lt(target_end)
    ].copy()
    future_purchases = future.loc[eligible_purchase_mask(future)]
    future_spend = future_purchases.groupby("customer_id")["line_amount"].sum()
    result = features.join(
        future_spend.rename("future_spend").reindex(features.index).fillna(0.0)
    )
    result.index.name = "customer_id"
    result["country"] = result["country"].astype(object)
    result.loc[result["country"].isna(), "country"] = np.nan

    if result[FEATURES].isna().all(axis=1).any():
        raise AssertionError("Se encontró un cliente sin ninguna característica disponible.")
    if (result["future_spend"] < 0).any():
        raise AssertionError("El objetivo debe ser no negativo bajo la regla de elegibilidad.")
    return result

train = make_customer_cohort("2010-01-01", "2010-07-01", "2010-07-01", "2010-12-10")
test = make_customer_cohort("2011-01-01", "2011-07-01", "2011-07-01", "2011-12-10")

cohort_summary = pd.DataFrame(
    [
        {
            "Conjunto": "Entrenamiento",
            "Clientes": len(train),
            "Clientes con gasto futuro > 0": int(train["future_spend"].gt(0).sum()),
            "Gasto futuro total": train["future_spend"].sum(),
            "Mediana del gasto futuro": train["future_spend"].median(),
        },
        {
            "Conjunto": "Prueba temporal",
            "Clientes": len(test),
            "Clientes con gasto futuro > 0": int(test["future_spend"].gt(0).sum()),
            "Gasto futuro total": test["future_spend"].sum(),
            "Mediana del gasto futuro": test["future_spend"].median(),
        },
    ]
)
display(cohort_summary)
print(f"Predictores numéricos: {NUMERIC_FEATURES}")
print(f"Predictores categóricos: {CATEGORICAL_FEATURES}")


,Conjunto,Clientes,Clientes con gasto futuro > 0,Gasto futuro total,Mediana del gasto futuro
0,Entrenamiento,2757,1932,"3,780,910.25",415.70
1,Prueba temporal,2724,1871,"3,832,576.06",390.83


Predictores numéricos: ['historical_spend', 'order_count', 'units_purchased', 'distinct_products', 'active_days', 'days_since_last_purchase', 'mean_order_spend', 'historical_return_lines', 'historical_cancellation_lines', 'historical_adjustment_lines']
Predictores categóricos: ['country']


## Preparación, entrenamiento y selección

Las imputaciones, el escalado numérico y la codificación one-hot se ajustan dentro de cada `Pipeline` usando **solo entrenamiento**; las categorías nuevas en prueba se ignoran de forma explícita. Se comparan un baseline de mediana, Ridge ajustado en la escala original y Random Forest sobre `log1p(y)` (con inversión `expm1`). Las predicciones negativas se limitan a cero, coherente con la definición del objetivo.

El criterio declarado para escoger un modelo es el menor MAE en la prueba temporal; RMSE penaliza errores grandes y R² sirve como referencia complementaria. Como la prueba también se usa para escoger entre candidatos, se debe reservar un periodo posterior para una evaluación definitiva antes de producción.


In [5]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_preprocessor(scale_numeric):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_pipeline = Pipeline(numeric_steps)
    categorical_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric_pipeline, NUMERIC_FEATURES),
            ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
    )

ridge = Pipeline(
    [
        ("preprocessor", make_preprocessor(scale_numeric=True)),
        ("model", Ridge(alpha=1000.0)),
    ]
)
forest = TransformedTargetRegressor(
    regressor=Pipeline(
        [
            ("preprocessor", make_preprocessor(scale_numeric=False)),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=300,
                    min_samples_leaf=3,
                    max_features=0.8,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    func=np.log1p,
    inverse_func=np.expm1,
)

models = {
    "Baseline (mediana)": DummyRegressor(strategy="median"),
    "Ridge": ridge,
    "Random Forest": forest,
}

X_train, y_train = train[FEATURES], train["future_spend"].astype(float)
X_test, y_test = test[FEATURES], test["future_spend"].astype(float)
assert len(X_train) == len(y_train) and len(X_test) == len(y_test)
assert set(FEATURES).isdisjoint({"customer_id", "future_spend"})

fitted_models = {}
predictions = {}
metric_rows = []
for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    prediction = np.maximum(estimator.predict(X_test), 0.0)
    fitted_models[name] = estimator
    predictions[name] = prediction
    metric_rows.append(
        {
            "Modelo": name,
            "MAE": mean_absolute_error(y_test, prediction),
            "RMSE": np.sqrt(mean_squared_error(y_test, prediction)),
            "R²": r2_score(y_test, prediction),
        }
    )

metrics = pd.DataFrame(metric_rows).sort_values(["MAE", "RMSE"]).reset_index(drop=True)
selected_name = metrics.loc[0, "Modelo"]
selected_model = fitted_models[selected_name]
display(metrics)
print(f"Modelo seleccionado por menor MAE temporal: {selected_name}")


,Modelo,MAE,RMSE,R²
0,Ridge,963.77,"4,928.33",0.48
1,Random Forest,"1,019.27","5,754.98",0.29
2,Baseline (mediana),"1,324.92","6,891.84",-0.02


Modelo seleccionado por menor MAE temporal: Ridge


In [6]:
selected_prediction = predictions[selected_name]
comparison = pd.DataFrame(
    {
        "Gasto real": y_test.to_numpy(),
        "Gasto predicho": selected_prediction,
        "Error absoluto": np.abs(y_test.to_numpy() - selected_prediction),
    },
    index=test.index,
)
print("Resumen de errores del modelo seleccionado, en la escala del gasto:")
display(comparison[["Gasto real", "Gasto predicho", "Error absoluto"]].describe().T)

print(
    f"MAE = {metrics.loc[metrics['Modelo'].eq(selected_name), 'MAE'].iloc[0]:,.2f}; "
    f"RMSE = {metrics.loc[metrics['Modelo'].eq(selected_name), 'RMSE'].iloc[0]:,.2f}; "
    f"R² = {metrics.loc[metrics['Modelo'].eq(selected_name), 'R²'].iloc[0]:.3f}."
)


Resumen de errores del modelo seleccionado, en la escala del gasto:


,count,mean,std,min,25%,50%,75%,max
Gasto real,"2,724.00","1,406.97","6,821.43",0.00,0.00,390.83,"1,150.80","168,469.60"
Gasto predicho,"2,724.00","1,314.21","3,679.40",1.85,259.61,591.33,"1,262.20","92,718.51"
Error absoluto,"2,724.00",963.77,"4,834.06",0.02,168.40,376.22,793.66,"168,442.06"


MAE = 963.77; RMSE = 4,928.33; R² = 0.478.


## Lectura de resultados y límites

La tabla de métricas permite comparar el error típico en unidades monetarias (MAE), la sensibilidad a errores grandes (RMSE) y la variación explicada respecto a predecir la media (R²). El baseline de mediana indica cuánto aporta el modelado frente a una predicción constante. El notebook muestra los valores calculados al ejecutarse; deben interpretarse junto con la distribución sesgada del gasto y la proporción de clientes con gasto cero.

`selected_model` queda disponible en memoria para inspección o predicción durante esta sesión. El notebook no guarda ni sobrescribe datos o artefactos de modelo. Antes de usarlo operativamente, revisar la definición de gasto con el área financiera, validar clientes nuevos (no cubiertos por esta cohorte), evaluar estabilidad por país y periodo, y medir el rendimiento en un periodo futuro que no haya intervenido en la selección.
